In [78]:
# Run once
!pip install pandas scikit-learn nltk

In [79]:
import pandas as pd
import numpy as np
import re
import nltk

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('punkt')
nltk.download('stopwords')

from nltk.corpus import stopwords

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [80]:
# Check columns
# Fix: Initialize df if it's not already defined (placeholder for actual data loading)
try:
    df.columns # This will raise NameError if df is not defined
except NameError:
    print("WARNING: 'df' is not defined. Creating a placeholder DataFrame. Please replace this with your actual data loading code.")
    # Creating a dummy DataFrame with 'title' and 'abstract' columns as expected by subsequent code
    data = {
        'title': ['Deep Learning for Image Classification', 'Natural Language Processing Advances', 'Computer Vision Trends', 'Robotics and AI', 'Data Science Applications'],
        'abstract': [
            'A comprehensive review of deep learning techniques applied to image classification tasks, covering CNNs, transfer learning, and adversarial examples.',
            'Exploring the latest advancements in NLP, including large language models, transformers, and their applications in text generation and sentiment analysis.',
            'An overview of current and emerging trends in computer vision, focusing on object detection, segmentation, and 3D reconstruction.',
            'Recent developments in humanoid robotics, including locomotion, manipulation, and human-robot interaction using artificial intelligence.',
            'Diverse applications of data science across various industries, from predictive analytics in finance to personalized recommendations in e-commerce.'
        ]
    }
    df = pd.DataFrame(data)

print(df.columns)

# 🔧 Handle nested JSON (if needed)
def extract_text(x):
    if isinstance(x, dict):
        return x.get('text', '')
    return x

if 'title' in df.columns:
    df['title'] = df['title'].apply(extract_text)

if 'abstract' in df.columns:
    df['abstract'] = df['abstract'].apply(extract_text)

# Keep only required columns
# Ensure 'title' and 'abstract' columns exist before selection
current_cols = df.columns.tolist()
if 'title' in current_cols and 'abstract' in current_cols:
    df = df[['title', 'abstract']]
elif 'title' in current_cols:
    df = df[['title']]
    df['abstract'] = '' # Add empty abstract column if not present
elif 'abstract' in current_cols:
    df = df[['abstract']]
    df['title'] = '' # Add empty title column if not present
else:
    print("WARNING: Neither 'title' nor 'abstract' columns found in the DataFrame. Creating an empty DataFrame with these columns.")
    df = pd.DataFrame(columns=['title', 'abstract'])

# Clean
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print("After Cleaning:", df.shape)
df.head()

Index(['title', 'abstract', 'clean_text'], dtype='object')
After Cleaning: (5, 2)


/tmp/ipykernel_5963/3055437161.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.dropna(inplace=True)


,title,abstract
0,Robotics and AI,"Recent developments in humanoid robotics, incl..."
1,Deep Learning for Image Classification,A comprehensive review of deep learning techni...
2,Computer Vision Trends,An overview of current and emerging trends in ...
3,Data Science Applications,Diverse applications of data science across va...
4,Natural Language Processing Advances,"Exploring the latest advancements in NLP, incl..."


In [81]:
# 🔥 Prevent memory issues
# Fix: Ensure sample size does not exceed the DataFrame size unless replace=True
sample_size = min(len(df), 5000)
df = df.sample(sample_size, random_state=42)

print("Sampled Shape:", df.shape)

Sampled Shape: (5, 2)


In [82]:
import re
import nltk

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('punkt')
nltk.download('stopwords')

from nltk.corpus import stopwords

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [83]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\W', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df['clean_text'] = df['title'] + " " + df['abstract']
df['clean_text'] = df['clean_text'].apply(clean_text)

df.head()

,title,abstract,clean_text
1,Deep Learning for Image Classification,A comprehensive review of deep learning techni...,deep learning image classification comprehensi...
4,Natural Language Processing Advances,"Exploring the latest advancements in NLP, incl...",natural language processing advances exploring...
2,Computer Vision Trends,An overview of current and emerging trends in ...,computer vision trends overview current emergi...
0,Robotics and AI,"Recent developments in humanoid robotics, incl...",robotics ai recent developments humanoid robot...
3,Data Science Applications,Diverse applications of data science across va...,data science applications diverse applications...


In [84]:
vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df['clean_text'])

print("Vector shape:", X.shape)

Vector shape: (5, 68)


In [85]:
def search_papers(query, top_n=5):
    query = clean_text(query)
    query_vec = vectorizer.transform([query])

    similarity = cosine_similarity(query_vec, X).flatten()

    top_indices = similarity.argsort()[-top_n:][::-1]

    results = []
    for idx in top_indices:
        results.append({
            "title": df.iloc[idx]['title'],
            "abstract": df.iloc[idx]['abstract'],
            "score": similarity[idx]
        })

    return results

In [86]:
def display_results(results):
    for i, res in enumerate(results):
        print(f"\n🔹 Result {i+1}")
        print("Title:", res['title'])
        print("Score:", round(res['score'], 3))
        print("Abstract:", res['abstract'][:300], "...")

In [87]:
query = "deep learning for image classification"

results = search_papers(query)
display_results(results)


🔹 Result 1
Title: Deep Learning for Image Classification
Score: 0.808
Abstract: A comprehensive review of deep learning techniques applied to image classification tasks, covering CNNs, transfer learning, and adversarial examples. ...

🔹 Result 2
Title: Data Science Applications
Score: 0.0
Abstract: Diverse applications of data science across various industries, from predictive analytics in finance to personalized recommendations in e-commerce. ...

🔹 Result 3
Title: Robotics and AI
Score: 0.0
Abstract: Recent developments in humanoid robotics, including locomotion, manipulation, and human-robot interaction using artificial intelligence. ...

🔹 Result 4
Title: Computer Vision Trends
Score: 0.0
Abstract: An overview of current and emerging trends in computer vision, focusing on object detection, segmentation, and 3D reconstruction. ...

🔹 Result 5
Title: Natural Language Processing Advances
Score: 0.0
Abstract: Exploring the latest advancements in NLP, including large language models, t

In [88]:
def recommend_similar_papers(index, top_n=5):
    similarity = cosine_similarity(X[index], X).flatten()

    similar_indices = similarity.argsort()[-top_n-1:-1][::-1]

    for i, idx in enumerate(similar_indices):
        print(f"\n🔹 Similar Paper {i+1}")
        print("Title:", df.iloc[idx]['title'])
        print("Abstract:", df.iloc[idx]['abstract'][:300], "...")

In [89]:
# Example
# The DataFrame 'df' (and thus the matrix 'X') only has 5 rows, so valid indices are 0 to 4.
# Calling with index=10 results in an IndexError.
# Let's use a valid index, for example, 0.
recommend_similar_papers(0)


🔹 Similar Paper 1
Title: Data Science Applications
Abstract: Diverse applications of data science across various industries, from predictive analytics in finance to personalized recommendations in e-commerce. ...

🔹 Similar Paper 2
Title: Robotics and AI
Abstract: Recent developments in humanoid robotics, including locomotion, manipulation, and human-robot interaction using artificial intelligence. ...

🔹 Similar Paper 3
Title: Computer Vision Trends
Abstract: An overview of current and emerging trends in computer vision, focusing on object detection, segmentation, and 3D reconstruction. ...

🔹 Similar Paper 4
Title: Natural Language Processing Advances
Abstract: Exploring the latest advancements in NLP, including large language models, transformers, and their applications in text generation and sentiment analysis. ...


In [90]:
from nltk.tokenize import sent_tokenize

def summarize(text, num_sentences=2):
    sentences = sent_tokenize(text)

    if len(sentences) <= num_sentences:
        return text

    return " ".join(sentences[:num_sentences])

In [91]:
import nltk
nltk.download('punkt_tab')
print(summarize(df.iloc[0]['abstract']))

A comprehensive review of deep learning techniques applied to image classification tasks, covering CNNs, transfer learning, and adversarial examples.


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [92]:
def run_agent():
    print("\n📚 AI Research Agent (Offline)")
    print("Type 'exit' to quit\n")

    while True:
        query = input("🔍 Enter your query: ")

        if query.lower() == 'exit':
            break

        results = search_papers(query)
        display_results(results)

run_agent()


📚 AI Research Agent (Offline)
Type 'exit' to quit

🔍 Enter your query: Data Science Engineer Network

🔹 Result 1
Title: Data Science Applications
Score: 0.623
Abstract: Diverse applications of data science across various industries, from predictive analytics in finance to personalized recommendations in e-commerce. ...

🔹 Result 2
Title: Robotics and AI
Score: 0.0
Abstract: Recent developments in humanoid robotics, including locomotion, manipulation, and human-robot interaction using artificial intelligence. ...

🔹 Result 3
Title: Computer Vision Trends
Score: 0.0
Abstract: An overview of current and emerging trends in computer vision, focusing on object detection, segmentation, and 3D reconstruction. ...

🔹 Result 4
Title: Natural Language Processing Advances
Score: 0.0
Abstract: Exploring the latest advancements in NLP, including large language models, transformers, and their applications in text generation and sentiment analysis. ...

🔹 Result 5
Title: Deep Learning for Image Class